In [1]:
from gensim.models import Word2Vec

word2vec_model = Word2Vec.load("./unlabelled/10_epoch_embeddings_long_sentences.model")

In [ ]:
import matplotlib.pyplot as plt
plt.plot([1,2,3])

In [2]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

model_name = 'jbochi/madlad400-3b-mt'
#model = T5ForConditionalGeneration.from_pretrained(model_name, device_map=None)
tokenizer = T5Tokenizer.from_pretrained(model_name)


In [3]:
import torch

In [4]:

text = "<2en> 羅漢係我兄弟，他成日在正午放風箏，就算天陰陰。"
input_ids = tokenizer(text, return_tensors="pt").input_ids

In [5]:
input_ids

tensor([[   805,     38,    805,  55031,  76718,  73348,   3803, 130895,  68540,
           1088,  10900,   3612,   2834,   2185,   6517,  77343,  11783,  26434,
         253412,   1088,   5350,  18635,   3911, 201462, 201462,   1142,      2]])

In [6]:
madlad_embeddings = torch.load("madlad_embedding.pth")

In [7]:
madlad_embeddings["weight"]

tensor([[-1.5856e-01, -3.5794e-01,  6.4598e-01,  ..., -2.7512e-02,
          4.7947e-01, -2.1065e-01],
        [ 1.6151e+00, -1.3196e+00, -1.1277e+00,  ...,  1.3023e+00,
          1.0067e-01,  5.4038e-03],
        [ 8.8776e-02,  7.7090e-01, -1.2226e+00,  ...,  6.7889e-01,
          8.6144e-01, -4.7935e-01],
        ...,
        [-1.2001e+01,  2.9804e+01,  4.8532e+00,  ...,  1.0312e+01,
         -2.9909e+00,  1.1952e+01],
        [-1.2817e+01, -1.7005e+01, -1.2661e+01,  ..., -2.4770e+01,
          5.4046e+00,  3.7453e+00],
        [ 4.5549e+00, -4.4047e+00,  4.4078e+00,  ...,  2.4507e+00,
          2.1307e+01, -9.2011e+00]])

In [8]:

def get_madlad_embedding_layer():
    return madlad_embeddings # decoder and encoder uses the same embedding weights
get_madlad_embedding_layer()["weight"].dtype

torch.float32

In [9]:
madlad_embeddings["weight"][input_ids]

tensor([[[ 1.6555e-01,  1.1890e+00, -2.1539e+00,  ..., -4.0582e+00,
           1.0744e+00, -5.7736e-01],
         [-3.9575e+00,  2.3815e-03,  1.6395e+00,  ...,  2.5391e+00,
          -1.6154e+01, -5.2441e+00],
         [ 1.6555e-01,  1.1890e+00, -2.1539e+00,  ..., -4.0582e+00,
           1.0744e+00, -5.7736e-01],
         ...,
         [ 2.3343e+00,  3.2717e+01,  7.6224e+00,  ...,  2.3185e+00,
          -1.1847e+01, -1.5596e+00],
         [-9.1812e+00,  1.7173e+01,  5.6834e+00,  ..., -1.1631e+01,
           2.9282e+00,  2.7449e+00],
         [ 8.8776e-02,  7.7090e-01, -1.2226e+00,  ...,  6.7889e-01,
           8.6144e-01, -4.7935e-01]]])

In [10]:
def get_madlad_embedding(token: str, include_start_and_end_tokens=False) -> torch.Tensor:
    '''
    [:, -1, :] is the EOF token embedding
    '''
    input_ids = tokenizer(token, return_tensors="pt").input_ids
    
    if not include_start_and_end_tokens:
        input_ids = input_ids[input_ids != 2]
        input_ids = input_ids[input_ids != 805]
    
    embeddings = get_madlad_embedding_layer()["weight"][input_ids]
    
    return embeddings


start_token = (get_madlad_embedding(''))
embedding = get_madlad_embedding('<2en>')

embedding.shape, start_token.shape

(torch.Size([1, 1024]), torch.Size([0, 1024]))

In [11]:
from torch import nn
import datasets
from datasets import DatasetDict
gatitos: DatasetDict = datasets.load_dataset("google/smol", "gatitos__yue_zh") # type: ignore

In [13]:
token_translations: list[dict[str, list[str]]] = []
for i in range(gatitos['train']['src'].__len__()): 
    for s in gatitos['train']['src'][i].split('; '):
        for t in gatitos['train']['trgs'][i]:
            token_translations.append({
                'src': s,
                'trg': t
            })

In [14]:
token_translations

[{'src': '廣州話', 'trg': '广州话'},
 {'src': '廣州話', 'trg': '广东话'},
 {'src': '廣州話', 'trg': '粤语'},
 {'src': '廣東話', 'trg': '广州话'},
 {'src': '廣東話', 'trg': '广东话'},
 {'src': '廣東話', 'trg': '粤语'},
 {'src': '粵語', 'trg': '广州话'},
 {'src': '粵語', 'trg': '广东话'},
 {'src': '粵語', 'trg': '粤语'},
 {'src': '白話', 'trg': '广州话'},
 {'src': '白話', 'trg': '广东话'},
 {'src': '白話', 'trg': '粤语'},
 {'src': '我', 'trg': '我'},
 {'src': '一', 'trg': '一'},
 {'src': '係', 'trg': '是'},
 {'src': '你好', 'trg': '你好'},
 {'src': '得', 'trg': '行'},
 {'src': '你點啊', 'trg': '你怎样啊'},
 {'src': '你好', 'trg': '你好'},
 {'src': '好嘅', 'trg': '好的'},
 {'src': '唔得', 'trg': '不行'},
 {'src': '多謝你', 'trg': '谢谢你'},
 {'src': '個', 'trg': '那个'},
 {'src': '你', 'trg': '你'},
 {'src': '早晨', 'trg': '早上好'},
 {'src': '乜嘢', 'trg': '什么'},
 {'src': '點解', 'trg': '为什么'},
 {'src': '好', 'trg': '好'},
 {'src': '喺', 'trg': '在'},
 {'src': '你最近點啊', 'trg': '你最近怎么样啊'},
 {'src': '點樣', 'trg': '怎么样'},
 {'src': '去', 'trg': '去'},
 {'src': '好', 'trg': '好'},
 {'src': '所以', 'trg': '所以'},
 {'

In [23]:
len(token_translations)

5351

In [12]:
token_translations[0]["trg"]

'广州话'

In [15]:
import numpy as np

In [16]:
indices =np.where(np.array([x["trg"] in word2vec_model.wv for x in token_translations]) == True)

In [17]:
token_translations[int(indices[0][0])]["trg"] in word2vec_model.wv

True

In [17]:
test_token = token_translations[int(indices[0][0])]["trg"]

In [17]:
word2vec_model.wv[test_token]

array([ 0.01866808, -0.0612832 ,  0.10284466, ...,  0.00310392,
       -0.05295956,  0.11279958], shape=(1024,), dtype=float32)

In [18]:
get_madlad_embedding(test_token)

tensor([[ -1.7124,  19.3426, -12.8546,  ..., -14.0900,   9.3031,  -4.6456]])

In [18]:
torch.norm(torch.tensor(word2vec_model.wv[token_translations[int(indices[0][30])]["trg"]]) - get_madlad_embedding(token_translations[int(indices[0][30])]["src"])) ** 2

tensor(265471.6250)

In [21]:
token_translations[indices[0][30]]

{'src': '喺', 'trg': '在'}

In [22]:
get_madlad_embedding(token_translations[int(indices[0][1])]["src"])

tensor([[-19.9191, -10.8058,  15.6723,  ...,   2.8460, -21.4432,  -8.5228]])

In [173]:
len(token_translations)

5351

In [189]:
len([x for x in token_translations if x["trg"] in word2vec_model.wv])

2251

In [19]:
losses=[]

In [20]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [21]:
epochs = 10
lr = 10**-3

from tqdm.notebook import tqdm

#W = torch.randn((1024, 1024))
W = torch.eye(1024, requires_grad=True, device=device)
optim = torch.optim.Adam([W], lr=lr)
criterion = nn.MSELoss()
batchsize = 64


In [22]:
for epoch in tqdm(range(epochs)):
    epoch_loss = 0
    for i in range(0, len(token_translations), batchsize):
        existing_indices = [x for x in range(i, min(i+batchsize, len(token_translations))) if token_translations[x]["trg"] in word2vec_model.wv]
        #print(existing_indices)
        if token_translations[i]["trg"] in word2vec_model.wv:
            mandarin_embeddings = torch.stack([torch.mean(get_madlad_embedding(token_translations[x]['src']),dim=0) for x in existing_indices]).to(device)
            cantonese_embeddings = torch.stack([torch.tensor(word2vec_model.wv[token_translations[x]['trg']]) for x in existing_indices]).to(device)
            
            pred = cantonese_embeddings @ W
            targets = mandarin_embeddings
            diff = torch.norm(pred - targets,dim=1)
            loss = diff @ diff


            loss.backward()
            optim.step()
            optim.zero_grad()
            epoch_loss += loss.item()

    if epoch % 10 == 0:
        print(epoch_loss)
        losses.append(epoch_loss)
    
print(epoch_loss)

  0%|          | 0/10 [00:00<?, ?it/s]

83963536.625
76207785.125


In [23]:
import matplotlib.pyplot as plt
plt.plot(losses)

: 

In [ ]:
#import matplotlib.pyplot as plt
#plt.plot(np.convolve(losses, np.ones(5), 'valid')/1000)


: 

In [205]:
losses[5]

3692139.5

In [197]:
np.array(losses)

array([2257820.    , 1932012.5   , 2449876.5   , ..., 2082323.5   ,
        460635.1875, 1233797.25  ], shape=(7000,))

In [ ]:
torch.norm(pred - targets, dim=)

tensor(11496.4219, grad_fn=<DotBackward0>)

In [81]:
W

tensor([[ 0.6607,  0.0059,  0.0018,  ...,  0.0029,  0.0067, -0.0092],
        [ 0.0121,  0.6251,  0.0007,  ...,  0.0260, -0.0128, -0.0042],
        [-0.0044,  0.0013,  0.6262,  ..., -0.0013, -0.0079, -0.0263],
        ...,
        [ 0.0062,  0.0157, -0.0113,  ...,  0.6269, -0.0063, -0.0278],
        [ 0.0135,  0.0027, -0.0038,  ..., -0.0141,  0.5675, -0.0239],
        [-0.0013,  0.0101, -0.0058,  ..., -0.0247, -0.0278,  0.6487]],
       requires_grad=True)

In [ ]:
import numpy as np
with open('transformation_matrix.npy' , "wb") as f:
    #np.save(f, W.detach().numpy())

In [11]:
import numpy as np
with open('transformation_matrix.npy', 'rb') as f:
    transformation_matrix = np.load(f)

In [14]:
embeds_ = get_madlad_embedding("<2en> 你好", include_start_and_end_tokens=True)

In [15]:
generated = model.generate(inputs_embeds=embeds_)

In [16]:
tokenizer.decode(generated[0], skip_special_tokens=True)

'Hello.'

In [17]:
translation_token = get_madlad_embedding("<2en>")
text = "羅漢係我兄弟，他成日在正午放風箏，就算天陰陰"

for t in tokenizer('\b<2en>')['input_ids']:
    print(tokenizer.decode(t), t)

 805
 58531
<2en> 38
</s> 2


In [196]:
t = 10900
print(torch.norm(torch.tensor(transformation_matrix @ word2vec_model.wv[tokenizer.decode(t)]).unsqueeze(0)))
print(torch.norm(get_madlad_embedding_layer(model)(torch.Tensor([t]).int())))

tensor(6.4839)
tensor(296.9704, grad_fn=<LinalgVectorNormBackward0>)


In [191]:
torch.norm(get_madlad_embedding(tokenizer.decode(t)))

tensor(346.1240, grad_fn=<LinalgVectorNormBackward0>)

In [183]:
text = "<2en>羅漢係我兄弟，他成日在正午放風箏，就算天陰陰"
#translation_token = get_madlad_embedding(text)

token_embeddings = []
for t in tokenizer(text)['input_ids']:
    print(t)
    if tokenizer.decode(t) in word2vec_model.wv:
        print(1)
        token_embedding = torch.tensor(word2vec_model.wv[tokenizer.decode(t)] @ transformation_matrix).unsqueeze(0)
    else:
        print(2)
        token_embedding = get_madlad_embedding_layer(model)(torch.Tensor([t]).int())
        
    token_embeddings.append(torch.Tensor(token_embedding))
    
token_embeddings = torch.tensor(np.array([x.detach().cpu().numpy() for x in token_embeddings])).permute(1,0,2)

805
2
38
2
55031
1
76718
1
73348
1
3803
1
130895
1
68540
1
1088
1
10900
1
3612
1
2834
1
2185
1
6517
1
77343
1
11783
1
26434
1
253412
1
1088
1
5350
1
18635
1
3911
1
201462
1
201462
1
2
2


In [178]:
token_embeddings.shape

torch.Size([1, 25, 1024])

In [169]:
token_embeddings

tensor([[[ 1.6555e-01,  1.1890e+00, -2.1539e+00,  ..., -4.0582e+00,
           1.0744e+00, -5.7736e-01],
         [-3.9575e+00,  2.3815e-03,  1.6395e+00,  ...,  2.5391e+00,
          -1.6154e+01, -5.2441e+00],
         [ 1.5735e+01,  1.1179e+01,  5.3661e+00,  ...,  8.9747e+00,
          -1.1488e+01, -1.1491e+01],
         ...,
         [ 2.3343e+00,  3.2717e+01,  7.6224e+00,  ...,  2.3185e+00,
          -1.1847e+01, -1.5596e+00],
         [ 2.3343e+00,  3.2717e+01,  7.6224e+00,  ...,  2.3185e+00,
          -1.1847e+01, -1.5596e+00],
         [ 8.8776e-02,  7.7090e-01, -1.2226e+00,  ...,  6.7889e-01,
           8.6144e-01, -4.7935e-01]]])

In [159]:
model.encoder.embed_tokens(tokenizer(text,  return_tensors="pt").input_ids).shape

torch.Size([1, 25, 1024])

In [173]:
torch.all(token_embeddings == model.encoder.embed_tokens(tokenizer(text,  return_tensors="pt").input_ids))

tensor(True)

In [160]:
token_embeddings == model.encoder.embed_tokens(tokenizer(text,  return_tensors="pt").input_ids)

False

In [141]:
torch.tensor(np.array([x.detach().cpu().numpy() for x in token_embeddings]))

tensor([[[ 1.6555e-01,  1.1890e+00, -2.1539e+00,  ..., -4.0582e+00,
           1.0744e+00, -5.7736e-01]],

        [[-3.9575e+00,  2.3815e-03,  1.6395e+00,  ...,  2.5391e+00,
          -1.6154e+01, -5.2441e+00]],

        [[ 1.5735e+01,  1.1179e+01,  5.3661e+00,  ...,  8.9747e+00,
          -1.1488e+01, -1.1491e+01]],

        ...,

        [[ 2.3343e+00,  3.2717e+01,  7.6224e+00,  ...,  2.3185e+00,
          -1.1847e+01, -1.5596e+00]],

        [[ 2.3343e+00,  3.2717e+01,  7.6224e+00,  ...,  2.3185e+00,
          -1.1847e+01, -1.5596e+00]],

        [[ 8.8776e-02,  7.7090e-01, -1.2226e+00,  ...,  6.7889e-01,
           8.6144e-01, -4.7935e-01]]])

In [102]:
model.encoder.embed_tokens(tokenizer(text,  return_tensors="pt").input_ids)

tensor([[[ 1.6555e-01,  1.1890e+00, -2.1539e+00,  ..., -4.0582e+00,
           1.0744e+00, -5.7736e-01],
         [-3.9575e+00,  2.3815e-03,  1.6395e+00,  ...,  2.5391e+00,
          -1.6154e+01, -5.2441e+00],
         [ 1.5735e+01,  1.1179e+01,  5.3661e+00,  ...,  8.9747e+00,
          -1.1488e+01, -1.1491e+01],
         ...,
         [ 2.3343e+00,  3.2717e+01,  7.6224e+00,  ...,  2.3185e+00,
          -1.1847e+01, -1.5596e+00],
         [ 2.3343e+00,  3.2717e+01,  7.6224e+00,  ...,  2.3185e+00,
          -1.1847e+01, -1.5596e+00],
         [ 8.8776e-02,  7.7090e-01, -1.2226e+00,  ...,  6.7889e-01,
           8.6144e-01, -4.7935e-01]]], grad_fn=<EmbeddingBackward0>)

In [68]:
torch.tensor(token_embeddings)

ValueError: only one element tensors can be converted to Python scalars

In [54]:
torch.stack(token_embeddings).shape

torch.Size([25, 1, 1024])

In [58]:
torch.mean(torch.stack(token_embeddings), dim=0).shape

torch.Size([1, 1024])

In [59]:
output = model.generate(inputs_embeds=torch.mean(torch.stack(token_embeddings).unsqueeze(0), dim=0))

In [182]:
out = model.generate(inputs_embeds=token_embeddings)
tokenizer.decode(out[0], skip_special_tokens=True)

'I i k ets tn y k s y y t a t y y'

In [106]:
true_embeds = model.encoder.embed_tokens(tokenizer(text,return_tensors="pt").input_ids)

In [ ]:
true_embeds = model.encoder.embed_tokens(tokenizer(text,return_tensors="pt").input_ids)
z = model.generate(inputs_embeds=true_embeds)
tokenizer.decode(z[0], skip_special_tokens=True)

'Rohan is my brother, he flies kites at noon every day, even in the dark'

In [85]:
tokenizer.decode(out[0], skip_special_tokens=True)

'Rohan is my brother, he is a rocket, he is a rocket, he is'

In [81]:
z = model.generate(tokenizer(text, return_tensors="pt").input_ids.to(model.device))
tokenizer.decode(z[0], skip_special_tokens=True)

'Rohan is my brother, he flies kites at noon every day, even in the dark'